# Rollout Noise Inspection (Surface Theta/U/V/Salt)

This notebook loads a small subset of Ocean_Emulator training data and visualizes rollout-noise artifacts on **surface** fields:
- Theta
- U
- V
- Salt

For each field, you will see side-by-side:
1. Original
2. Noisy
3. Delta (noisy - original)
4. Structured front mask


In [ ]:
from __future__ import annotations

import dataclasses
import inspect
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Force local `src/ocean_emulators` imports even if an older package was
# already imported earlier in this kernel session.
for _mod_name in list(sys.modules):
    if _mod_name == "ocean_emulators" or _mod_name.startswith("ocean_emulators."):
        del sys.modules[_mod_name]

from ocean_emulators.config import TrainConfig
from ocean_emulators.constants import BOUNDARY_VARS, PROGNOSTIC_VARS
from ocean_emulators.datasets import TorchTrainDataset
from ocean_emulators.models.modules.rollout_noise import RolloutNoiseInjector
from ocean_emulators.utils.data import Masks
from ocean_emulators.utils.train import collate_raw_train_data

ROLL_OUT_INIT_PARAMS = set(inspect.signature(RolloutNoiseInjector.__init__).parameters)

plt.style.use("default")
plt.rcParams["figure.dpi"] = 120
print(f"Project root: {PROJECT_ROOT}")
print("RolloutNoiseInjector supports gaussian_mode:", "gaussian_mode" in ROLL_OUT_INIT_PARAMS)


In [ ]:
# -----------------------------
# Data loading configuration
# -----------------------------
TRAIN_CONFIG_PATH = PROJECT_ROOT / "configs" / "samudra_llc" / "train.yaml"
TIME_STEPS_TO_KEEP = 2
HIST_FOR_NOTEBOOK = 0
Y_SLICE = slice(0, 180)
X_SLICE = slice(744-180, 744)
MAX_BOUNDARY_CHANNELS = 2

cfg = TrainConfig.from_yaml_and_cli([str(TRAIN_CONFIG_PATH)])
prognostic_var_names = PROGNOSTIC_VARS[cfg.experiment.prognostic_vars_key]
boundary_var_names = BOUNDARY_VARS[cfg.experiment.boundary_vars_key]

def find_surface_var(all_vars: list[str], base_name: str) -> str:
    target = f"{base_name}_0"
    for var in all_vars:
        if var == target:
            return var
    raise ValueError(
        f"Could not find surface variable {target} in prognostic set. Available examples: {all_vars[:12]}"
    )

surface_var_by_field = {
    "Theta": find_surface_var(prognostic_var_names, "Theta"),
    "U": find_surface_var(prognostic_var_names, "U"),
    "V": find_surface_var(prognostic_var_names, "V"),
    "Salt": find_surface_var(prognostic_var_names, "Salt"),
}
selected_fields = ["Theta", "U", "V", "Salt"]
selected_prognostic_vars = [surface_var_by_field[f] for f in selected_fields]
selected_boundary_vars = boundary_var_names[:MAX_BOUNDARY_CHANNELS]
field_to_channel_index = {field: i for i, field in enumerate(selected_fields)}

print("Selected prognostic channels:")
for field in selected_fields:
    print(f"  {field}: {surface_var_by_field[field]}")

container = cfg.data.build(
    data_root=cfg.experiment.resolved_data_root,
    prognostic_var_names=prognostic_var_names,
    boundary_var_names=boundary_var_names,
)
source = container.source.slice(cfg.train_time)

def infer_spatial_dims(ds):
    for y_dim, x_dim in [("y", "x"), ("lat", "lon"), ("j", "i")]:
        if y_dim in ds.dims and x_dim in ds.dims:
            return y_dim, x_dim
    raise ValueError(f"Could not infer spatial dims from {list(ds.dims)}")

y_dim, x_dim = infer_spatial_dims(source.data)

def _subset_dataset(ds, include_time):
    indexers = {}
    if include_time and "time" in ds.dims:
        indexers["time"] = slice(0, TIME_STEPS_TO_KEEP)
    if y_dim in ds.dims:
        indexers[y_dim] = Y_SLICE
    if x_dim in ds.dims:
        indexers[x_dim] = X_SLICE
    return ds.isel(indexers) if indexers else ds

small_source = source.map(
    lambda data, means, stds: (
        _subset_dataset(data, include_time=True),
        _subset_dataset(means, include_time=False),
        _subset_dataset(stds, include_time=False),
    ),
    suffix="notebook_subset",
)
small_source = dataclasses.replace(
    small_source,
    masks=Masks(
        prognostic=source.masks.prognostic[: len(selected_prognostic_vars), Y_SLICE, X_SLICE],
        boundary=source.masks.boundary[Y_SLICE, X_SLICE],
    ),
)

train_dataset = TorchTrainDataset(
    src=small_source,
    prognostic_var_names=selected_prognostic_vars,
    boundary_var_names=selected_boundary_vars,
    hist=HIST_FOR_NOTEBOOK,
    steps=1,
    normalize_before_mask=cfg.data.normalize_before_mask,
    masked_fill_value=cfg.data.masked_fill_value,
    stride=1,
)

raw = train_dataset[0]
raw_batched = collate_raw_train_data([raw])
train_data = train_dataset.to_train_data(raw_batched)
base_input = train_data.get_initial_input()
base_prognostic = base_input[:, : train_dataset.num_prognostic_channels].detach()
wet_mask = train_dataset.wet[: train_dataset.num_prognostic_channels].detach().cpu()

print(f"Loaded sample shape: {tuple(base_prognostic.shape)} (batch, channel, H, W)")
print(f"Using spatial dims: {y_dim}, {x_dim}")


## Tunable Parameters

### Global lattice Gaussian component
- `gaussian_std`: overall amplitude of global additive noise.
- `gaussian_lattice_stride`: size scale of coherent lattice-connected regions (bigger = larger patches).
- `gaussian_blur_kernel`, `gaussian_blur_sigma`: smoothing of lattice patterns so patch centers are stronger and edges softer.

### Frontal patch-error component
- `structured_scale`: amplitude of front-focused errors.
- `front_power`: concentrates emphasis toward strongest-gradient fronts.
- `smoothing_kernel`: smooths the front mask itself.
- `structured_seed_prob`: how often frontal seeds are placed (higher = more error patches).
- `structured_patch_kernel`, `structured_patch_sigma`: control patch footprint/size around seeds.
- `structured_patch_quantile`: keeps only highest-magnitude patch activations (higher = fewer, more extreme patches).
- `sign_mode`:
  - `patch`: sign follows coherent patch seeds (used for all presets).
  - `batch`: all errors in a sample are globally high or globally low.
  - `pixel`: per-pixel sign flips (least coherent).

### Shared
- `probability`: per-sample probability of applying any noise at all.
- `seed`: random seed for reproducible tuning in the notebook.

### Presets
Use the preset controls to instantly load:
- `low`: weakest noise injection.
- `medium`: your requested baseline profile.
- `high`: strongest noise injection.

## Structured Front Mask
The mask is derived from spatial gradients of the prognostic tensor:
1. Compute x/y finite differences per channel.
2. Take gradient magnitude and average across channels.
3. Normalize by per-sample max gradient.
4. Apply `front_power` and smoothing (`smoothing_kernel`).

Interpretation: bright mask values mark sharp fronts where model errors tend to be amplified. The frontal patch error component places sparse seeds there, then grows them into connected high-error patches.


In [ ]:
field_cmap = {
    "Theta": "Spectral_r",
    "U": "bwr",
    "V": "bwr",
    "Salt": "viridis",
}

_missing_arg_warning_shown = False

def _build_injector_compat(**kwargs):
    global _missing_arg_warning_shown
    supported_kwargs = {k: v for k, v in kwargs.items() if k in ROLL_OUT_INIT_PARAMS}
    dropped = sorted(set(kwargs) - set(supported_kwargs))
    if dropped and not _missing_arg_warning_shown:
        print(
            "Note: current RolloutNoiseInjector does not expose these args; ignoring:",
            ", ".join(dropped),
        )
        _missing_arg_warning_shown = True
    return RolloutNoiseInjector(**supported_kwargs)


def plot_noise_manifestation(
    gaussian_std=0.03,
    gaussian_lattice_stride=8,
    gaussian_blur_kernel=7,
    gaussian_blur_sigma=2.0,
    structured_scale=0.08,
    probability=1.0,
    front_power=1.5,
    smoothing_kernel=7,
    structured_seed_prob=0.02,
    structured_patch_kernel=19,
    structured_patch_sigma=4.0,
    structured_patch_quantile=0.97,
    sign_mode="patch",
    seed=0,
):
    device = base_prognostic.device
    injector = _build_injector_compat(
        wet=wet_mask.to(device),
        probability=float(probability),
        gaussian_std=float(gaussian_std),
        gaussian_mode="lattice",
        gaussian_lattice_stride=int(gaussian_lattice_stride),
        gaussian_blur_kernel=int(gaussian_blur_kernel),
        gaussian_blur_sigma=float(gaussian_blur_sigma),
        structured_scale=float(structured_scale),
        structured_front_power=float(front_power),
        structured_mask_smoothing_kernel=int(smoothing_kernel),
        structured_seed_prob=float(structured_seed_prob),
        structured_patch_kernel=int(structured_patch_kernel),
        structured_patch_sigma=float(structured_patch_sigma),
        structured_patch_quantile=float(structured_patch_quantile),
        structured_sign_mode=str(sign_mode),
    )
    injector.train()

    torch.manual_seed(int(seed))
    with torch.no_grad():
        noisy = injector(base_prognostic.clone())
        delta = noisy - base_prognostic
        if hasattr(injector, "_front_mask"):
            front_mask = injector._front_mask(base_prognostic).detach()
        else:
            front_mask = torch.zeros(
                (base_prognostic.shape[0], 1, base_prognostic.shape[-2], base_prognostic.shape[-1]),
                device=base_prognostic.device,
                dtype=base_prognostic.dtype,
            )

    fig, axes = plt.subplots(
        len(selected_fields), 4, figsize=(18, 3.8 * len(selected_fields)), constrained_layout=True
    )

    column_titles = [
        "Original",
        "Noisy",
        "Delta (noisy - original)",
        "Structured front mask",
    ]
    for c, title in enumerate(column_titles):
        axes[0, c].set_title(title)

    for r, field in enumerate(selected_fields):
        idx = field_to_channel_index[field]
        cmap_field = field_cmap[field]

        original_2d = base_prognostic[0, idx].detach().cpu().numpy()
        noisy_2d = noisy[0, idx].detach().cpu().numpy()
        delta_2d = delta[0, idx].detach().cpu().numpy()
        front_2d = front_mask[0, 0].detach().cpu().numpy()

        vmin = min(float(original_2d.min()), float(noisy_2d.min()))
        vmax = max(float(original_2d.max()), float(noisy_2d.max()))
        delta_abs_max = max(1e-8, float(np.abs(delta_2d).max()))

        im0 = axes[r, 0].imshow(original_2d, cmap=cmap_field, vmin=vmin, vmax=vmax)
        im1 = axes[r, 1].imshow(noisy_2d, cmap=cmap_field, vmin=vmin, vmax=vmax)
        im2 = axes[r, 2].imshow(delta_2d, cmap="RdBu_r", vmin=-delta_abs_max, vmax=delta_abs_max)
        im3 = axes[r, 3].imshow(front_2d, cmap="magma", vmin=0.0, vmax=1.0)

        fig.colorbar(im0, ax=axes[r, 0], fraction=0.046, pad=0.03)
        fig.colorbar(im1, ax=axes[r, 1], fraction=0.046, pad=0.03)
        fig.colorbar(im2, ax=axes[r, 2], fraction=0.046, pad=0.03)
        fig.colorbar(im3, ax=axes[r, 3], fraction=0.046, pad=0.03)

        axes[r, 0].set_ylabel(field, rotation=90)
        for c in range(4):
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])

    fig.suptitle(
        f"g_std={gaussian_std}, g_stride={gaussian_lattice_stride}, g_blur=({gaussian_blur_kernel},{gaussian_blur_sigma}), s_scale={structured_scale}, p={probability}, front_power={front_power}, seed_prob={structured_seed_prob}, patch=({structured_patch_kernel},{structured_patch_sigma}), q={structured_patch_quantile}, sign={sign_mode}, seed={seed}",
        fontsize=10,
    )
    plt.show()


In [ ]:
PRESETS = {
    "low": {
        "gaussian_std": 0.005,
        "gaussian_lattice_stride": 4,
        "gaussian_blur_kernel": 5,
        "gaussian_blur_sigma": 1.6,
        "structured_scale": 0.05,
        "probability": 0.50,
        "front_power": 1.0,
        "smoothing_kernel": 3,
        "structured_seed_prob": 0.005,
        "structured_patch_kernel": 13,
        "structured_patch_sigma": 4.0,
        "structured_patch_quantile": 0.90,
        "sign_mode": "patch",
        "seed": 42,
    },
    "medium": {
        "gaussian_std": 0.01,
        "gaussian_lattice_stride": 3,
        "gaussian_blur_kernel": 5,
        "gaussian_blur_sigma": 2.0,
        "structured_scale": 0.10,
        "probability": 0.75,
        "front_power": 1.0,
        "smoothing_kernel": 3,
        "structured_seed_prob": 0.01,
        "structured_patch_kernel": 15,
        "structured_patch_sigma": 5.2,
        "structured_patch_quantile": 0.80,
        "sign_mode": "patch",
        "seed": 42,
    },
    "high": {
        "gaussian_std": 0.03,
        "gaussian_lattice_stride": 3,
        "gaussian_blur_kernel": 7,
        "gaussian_blur_sigma": 2.4,
        "structured_scale": 0.20,
        "probability": 1.00,
        "front_power": 1.4,
        "smoothing_kernel": 3,
        "structured_seed_prob": 0.03,
        "structured_patch_kernel": 19,
        "structured_patch_sigma": 6.0,
        "structured_patch_quantile": 0.65,
        "sign_mode": "patch",
        "seed": 42,
    },
}

preset_dropdown = widgets.Dropdown(
    options=["custom", "low", "medium", "high"],
    value="medium",
    description="preset",
)
apply_preset_button = widgets.Button(description="Apply preset")


gaussian_slider = widgets.FloatSlider(value=0.03, min=0.0, max=0.2, step=0.005, description="gaussian")
lattice_stride_slider = widgets.IntSlider(value=8, min=3, max=20, step=1, description="g_stride")
gaussian_kernel_slider = widgets.SelectionSlider(options=[1, 3, 5, 7, 9, 11], value=7, description="g_kernel")
gaussian_sigma_slider = widgets.FloatSlider(value=2.0, min=0.0, max=8.0, step=0.1, description="g_sigma")

structured_slider = widgets.FloatSlider(value=0.08, min=0.0, max=0.35, step=0.005, description="s_scale")
prob_slider = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description="probability")
front_power_slider = widgets.FloatSlider(value=1.5, min=0.1, max=4.0, step=0.1, description="front_power")
mask_kernel_slider = widgets.SelectionSlider(options=[1, 3, 5, 7, 9, 11], value=7, description="mask_kernel")
seed_prob_slider = widgets.FloatSlider(value=0.02, min=0.001, max=0.2, step=0.001, description="seed_prob")
patch_kernel_slider = widgets.SelectionSlider(options=[7, 11, 13, 15, 19, 23, 27], value=19, description="patch_kernel")
patch_sigma_slider = widgets.FloatSlider(value=4.0, min=0.5, max=8.0, step=0.1, description="patch_sigma")
patch_quantile_slider = widgets.FloatSlider(value=0.97, min=0.5, max=0.99, step=0.01, description="patch_q")
sign_dropdown = widgets.Dropdown(options=["patch", "batch", "pixel"], value="patch", description="sign_mode")
seed_slider = widgets.IntSlider(value=0, min=0, max=9999, step=1, description="seed")


def _apply_selected_preset(_=None):
    name = preset_dropdown.value
    if name == "custom":
        return
    params = PRESETS[name]
    gaussian_slider.value = params["gaussian_std"]
    lattice_stride_slider.value = params["gaussian_lattice_stride"]
    gaussian_kernel_slider.value = params["gaussian_blur_kernel"]
    gaussian_sigma_slider.value = params["gaussian_blur_sigma"]
    structured_slider.value = params["structured_scale"]
    prob_slider.value = params["probability"]
    front_power_slider.value = params["front_power"]
    mask_kernel_slider.value = params["smoothing_kernel"]
    seed_prob_slider.value = params["structured_seed_prob"]
    patch_kernel_slider.value = params["structured_patch_kernel"]
    patch_sigma_slider.value = params["structured_patch_sigma"]
    patch_quantile_slider.value = params["structured_patch_quantile"]
    sign_dropdown.value = params["sign_mode"]
    seed_slider.value = params["seed"]


apply_preset_button.on_click(_apply_selected_preset)
_apply_selected_preset()

preset_controls = widgets.HBox([preset_dropdown, apply_preset_button])
controls = widgets.VBox([
    preset_controls,
    gaussian_slider,
    lattice_stride_slider,
    gaussian_kernel_slider,
    gaussian_sigma_slider,
    structured_slider,
    prob_slider,
    front_power_slider,
    mask_kernel_slider,
    seed_prob_slider,
    patch_kernel_slider,
    patch_sigma_slider,
    patch_quantile_slider,
    sign_dropdown,
    seed_slider,
])

out = widgets.interactive_output(
    plot_noise_manifestation,
    {
        "gaussian_std": gaussian_slider,
        "gaussian_lattice_stride": lattice_stride_slider,
        "gaussian_blur_kernel": gaussian_kernel_slider,
        "gaussian_blur_sigma": gaussian_sigma_slider,
        "structured_scale": structured_slider,
        "probability": prob_slider,
        "front_power": front_power_slider,
        "smoothing_kernel": mask_kernel_slider,
        "structured_seed_prob": seed_prob_slider,
        "structured_patch_kernel": patch_kernel_slider,
        "structured_patch_sigma": patch_sigma_slider,
        "structured_patch_quantile": patch_quantile_slider,
        "sign_mode": sign_dropdown,
        "seed": seed_slider,
    },
)
display(controls, out)
